# RQ1: Is higher adoption of aggregated security practices associated with measurable improvements in security outcomes? 



## Imports

In [ ]:
import math
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from matplotlib import pyplot as plt

from settings import load_settings
from capstone import construct_final_dataset as fd

## Constants

In [ ]:
SETTINGS = load_settings()

In [ ]:
CONTROL_COLUMNS = [
    fd.C_REPOSITORY_CONTRIBUTIONS_COUNT,
    fd.C_REPOSITORY_SIZE_IN_KB,
    fd.C_REPOSITORY_AGE_IN_YEARS,
    fd.C_REPOSITORY_COMMIT_STALENESS_IN_DAYS,
    fd.C_PACKAGE_DEPENDENCY_COUNT,
    fd.C_PACKAGE_TOTAL_DOWNLOADS,
]

TARGET_COLUMNS = [
    fd.T_VULNERABILITY_COUNT,
    fd.T_MTTU,
    fd.T_MTTR,
]

C_LOG_REPOSITORY_CONTRIBUTIONS_COUNT = f"log_{fd.C_REPOSITORY_CONTRIBUTIONS_COUNT}"
C_LOG_REPOSITORY_SIZE_IN_KB = f"log_{fd.C_REPOSITORY_SIZE_IN_KB}"
C_LOG_REPOSITORY_AGE_IN_YEARS = f"log_{fd.C_REPOSITORY_AGE_IN_YEARS}"
C_LOG_REPOSITORY_COMMIT_STALENESS_IN_DAYS = f"log_{fd.C_REPOSITORY_COMMIT_STALENESS_IN_DAYS}"
C_LOG_PACKAGE_DEPENDENCY_COUNT = f"log_{fd.C_PACKAGE_DEPENDENCY_COUNT}"
C_LOG_PACKAGE_TOTAL_DOWNLOADS = f"log_{fd.C_PACKAGE_TOTAL_DOWNLOADS}"

T_LOG_MTTU = f"log_{fd.T_MTTU}"
T_LOG_MTTR = f"log_{fd.T_MTTR}"


LOG_CONTROL_COLUMNS = [
    C_LOG_REPOSITORY_CONTRIBUTIONS_COUNT,
    C_LOG_REPOSITORY_SIZE_IN_KB,
    C_LOG_REPOSITORY_AGE_IN_YEARS,
    C_LOG_REPOSITORY_COMMIT_STALENESS_IN_DAYS,
    C_LOG_PACKAGE_DEPENDENCY_COUNT,
    C_LOG_PACKAGE_TOTAL_DOWNLOADS,
]

## Loading Data

In [ ]:
raw_df = pl.read_parquet(SETTINGS.research_question_1_dataset_path)

In [ ]:
raw_df.head()

In [ ]:
raw_df.describe()

## Inspecting Data

In [ ]:
def plot_distributions(data: pd.DataFrame, columns: list[str]) -> None:
    ncols = math.ceil(len(columns) ** 0.5)
    nrows = math.ceil(len(columns) / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 8))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.histplot(data[col].dropna(), kde=True, ax=axes[i])
        axes[i].set_title(col)

    # remove empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

### Control Variables Distributions

In [ ]:
plot_distributions(raw_df.to_pandas(), CONTROL_COLUMNS)

### Target Variables Distributions

In [ ]:
plot_distributions(raw_df.to_pandas(), TARGET_COLUMNS)

In [ ]:
for col in [fd.T_MTTR, fd.T_MTTU]:
    s = raw_df.to_pandas()[col]
    print(col)
    print("min:", s.min())
    print("zeros:", (s == 0).sum())
    print("nulls:", s.isna().sum())
    print()

## Transforming Data

In [ ]:
df = (
    raw_df
        .drop_nulls()
        .drop_nans()
        .to_pandas()
)

df[C_LOG_REPOSITORY_CONTRIBUTIONS_COUNT] = np.log1p(df[fd.C_REPOSITORY_CONTRIBUTIONS_COUNT])
df[C_LOG_REPOSITORY_SIZE_IN_KB] = np.log1p(df[fd.C_REPOSITORY_SIZE_IN_KB])
df[C_LOG_REPOSITORY_AGE_IN_YEARS] = np.log1p(df[fd.C_REPOSITORY_AGE_IN_YEARS])
df[C_LOG_REPOSITORY_COMMIT_STALENESS_IN_DAYS] = np.log1p(df[fd.C_REPOSITORY_COMMIT_STALENESS_IN_DAYS])
df[C_LOG_PACKAGE_DEPENDENCY_COUNT] = np.log1p(df[fd.C_PACKAGE_DEPENDENCY_COUNT])
df[C_LOG_PACKAGE_TOTAL_DOWNLOADS] = np.log1p(df[fd.C_PACKAGE_TOTAL_DOWNLOADS])

df[T_LOG_MTTR] = np.log1p(df[fd.T_MTTR])
df[T_LOG_MTTU] = np.log1p(df[fd.T_MTTU])

In [ ]:
df.describe()

In [ ]:
plot_distributions(df, LOG_CONTROL_COLUMNS)

In [ ]:
plot_distributions(df, [T_LOG_MTTR, T_LOG_MTTU])

## Modeling

We employ different modeling strategies tailored to the distributional properties of each outcome variable. For vulnerability counts, we begin with Poisson regression as a baseline but observe substantial overdispersion, with the variance exceeding the mean. To address this, we adopt Negative Binomial regression, which introduces a dispersion parameter to account for excess variability and provides a better fit for count data.

In contrast, mean time to recover (MTTR) and mean time to update (MTTU) are continuous, strictly positive duration variables rather than counts. Therefore, count-based models are not appropriate. Instead, we model these outcomes using generalized linear models with a Gamma distribution and log link, which are well-suited for right-skewed, positive data. As a baseline specification, we also estimate log-transformed ordinary least squares (OLS) models to ensure robustness and facilitate interpretation.

For all models, we log-transform highly skewed control variables (e.g., repository size, dependency count, and downloads) to reduce skewness and improve numerical stability. Coefficients from both Negative Binomial and Gamma models are interpreted as multiplicative effects, allowing us to express results in terms of percentage changes in the expected outcome.

In [ ]:
df.columns

### Target: vul_count

#### Baseline: Poisson Regression

In [ ]:
vul_count_formula = f"vul_count ~ aggregated_score + {' + '.join(LOG_CONTROL_COLUMNS)}"
vul_count_formula

In [ ]:
vul_count_poisson_model = smf.glm(formula=vul_count_formula, data=df, family=sm.families.Poisson()).fit()

In [ ]:
print(vul_count_poisson_model.summary())

In [ ]:
dispersion = vul_count_poisson_model.pearson_chi2 / vul_count_poisson_model.df_resid
print(f"Dispersion: {dispersion}")
if dispersion > 1.5:
    print("Evidence of overdispersion detected.")

#### Main: Negative Binomial Regression

In [ ]:
def fit_vul_count_nb_model(data: pd.DataFrame, target_variable: str):
    X = data[["aggregated_score", *LOG_CONTROL_COLUMNS]]
    X = sm.add_constant(X)
    y = data[target_variable]
    model = sm.NegativeBinomial(y, X).fit()
    return model

In [ ]:
vul_count_model = fit_vul_count_nb_model(df, fd.T_VULNERABILITY_COUNT)
print(vul_count_model.summary())

### Target: MTTU

#### Baseline: OLS Regression

In [ ]:
mttu_ols_formula = f"log_mttu ~ aggregated_score + {' + '.join(LOG_CONTROL_COLUMNS)}"
mttu_ols_formula

In [ ]:
mttu_ols_model = smf.ols(formula=mttu_ols_formula, data=df).fit()

print(mttu_ols_model.summary())

#### Main: Gamma GLM Regression

In [ ]:
mttu_gamma_formula =  f"mttu ~ aggregated_score + {' + '.join(LOG_CONTROL_COLUMNS)}"

In [ ]:
mttu_df = df.loc[df["mttu"] > 0].copy()
mttu_gamma_model = smf.glm(
    formula=mttu_gamma_formula,
    data=mttu_df,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(mttu_gamma_model.summary())

In [ ]:
# Lower AIC = better fit
print("OLS AIC:", mttu_ols_model.aic)
print("Gamma AIC:", mttu_gamma_model.aic)

In [ ]:
mttu_model = mttu_ols_model if mttu_ols_model.aic < mttu_gamma_model.aic else mttu_gamma_model

### Target: MTTR

#### Baseline: OLS Regression

In [ ]:
mttr_ols_formula = f"log_mttr ~ aggregated_score + {' + '.join(LOG_CONTROL_COLUMNS)}"
mttr_ols_formula

In [ ]:
mttr_ols_model = smf.ols(formula=mttr_ols_formula, data=df).fit()
print(mttr_ols_model.summary())

#### Main: Gamma GLM Regression

In [ ]:
mttr_gamma_formula =  f"mttr ~ aggregated_score + {' + '.join(LOG_CONTROL_COLUMNS)}"
mttr_gamma_formula

In [ ]:
mttr_df = df.loc[df["mttr"] > 0].copy()

mttr_gamma_model = smf.glm(
    formula=mttr_gamma_formula,
    data=mttr_df,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(mttr_gamma_model.summary())

In [ ]:
# Lower AIC = better fit
print("OLS AIC:", mttr_ols_model.aic)
print("Gamma AIC:", mttr_gamma_model.aic)

In [ ]:
mttr_model = mttr_gamma_model

### Summary

In [ ]:
stargazer = Stargazer([
    vul_count_model,
    mttr_model, 
    mttu_model,
])

stargazer.title("Regression Results")
stargazer.custom_columns(["Vul Count", "MTTR", "MTTU"], [1, 1, 1])
stargazer.show_confidence_intervals(True)

In [ ]:
stargazer